In [13]:
from opacus.accountants.analysis.rdp import compute_rdp
import math

def calibrate_sigma_rdp_poisson(*, alpha, eps_alpha, sample_rate, steps, iters=60):
    """
    Find the smallest sigma such that RDP at order alpha is <= eps_alpha
    for subsampled Gaussian (Poisson) with sampling rate q=sample_rate over `steps` steps.
    """
    return math.sqrt((steps * alpha) / (2.0 * eps_alpha))
    # def eps_at_sigma(sigma: float) -> float:
    #     return float(compute_rdp(
    #         q=sample_rate,
    #         noise_multiplier=sigma,
    #         steps=steps,
    #         orders=[alpha],
    #     )[0])

    # hi = 1.0
    # while eps_at_sigma(hi) > eps_alpha:
    #     hi *= 2.0

    # lo = 0.0
    # for _ in range(iters):
    #     mid = (lo + hi) / 2.0
    #     mid = max(mid, 1e-12)
    #     if eps_at_sigma(mid) > eps_alpha:
    #         lo = mid
    #     else:
    #         hi = mid
    # return hi

    import math

def sigma_from_rdp_fullbatch(*, alpha, eps_alpha, steps):
    """
    q = 1 RDP-Gaussian mechanism:
      eps_alpha = steps * alpha / (2 * sigma^2)
    """
    return math.sqrt((steps * alpha) / (2.0 * eps_alpha))



In [14]:
import math

def sigma_from_rdp_fullbatch(*, alpha: float, eps_alpha: float, steps: int) -> float:
    """
    q = 1 (no subsampling), Gaussian mechanism composed `steps` times:
      eps_alpha = steps * alpha / (2 * sigma^2)
    """
    if alpha <= 1:
        raise ValueError("RDP order alpha must be > 1")
    if eps_alpha <= 0:
        raise ValueError("eps_alpha must be > 0")
    if steps <= 0:
        raise ValueError("steps must be >= 1")
    return math.sqrt((steps * alpha) / (2.0 * eps_alpha))


In [15]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from opacus import PrivacyEngine

def dpsgd_rdp_run_fullbatch(
    base_model,
    dataset,
    *,
    canary_xy,
    alpha, eps_alpha,
    max_grad_norm,
    epochs,
    lr,
    device,
    noise_multiplier=None,
    loss_reduction="sum",
    return_train_losses=False,
):
    model = copy.deepcopy(base_model).to(device)

    criterion = nn.CrossEntropyLoss(reduction=loss_reduction)
    optimizer = optim.SGD(model.parameters(), lr=lr)

    # Full batch: one batch per epoch, matches q = 1 style
    train_loader = DataLoader(dataset, batch_size=len(dataset), shuffle=False, drop_last=False)

    q = 1.0
    steps_planned = int(epochs)

    sigma = noise_multiplier
    if sigma is None:
        sigma = sigma_from_rdp_fullbatch(alpha=alpha, eps_alpha=eps_alpha, steps=steps_planned)

    privacy_engine = PrivacyEngine(accountant="rdp")
    model, optimizer, private_loader = privacy_engine.make_private(
        module=model,
        optimizer=optimizer,
        data_loader=train_loader,
        noise_multiplier=sigma,
        max_grad_norm=max_grad_norm,
        poisson_sampling=True,     # with q=1 and one batch, this is effectively "take all"
        loss_reduction=loss_reduction,
    )

    train_losses = [] if return_train_losses else None
    steps_done = 0

    model.train()
    for _ in range(epochs):
        for X, y in private_loader:
            # With q=1 this should never be empty, but keep it harmlessly:
            if X.numel() == 0:
                continue

            X, y = X.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()

            steps_done += 1
            if return_train_losses:
                train_losses.append(float(loss.detach().cpu()))

    # Canary loss
    x_c, y_c = canary_xy
    model.eval()
    with torch.no_grad():
        canary_loss = criterion(
            model(x_c.to(device).unsqueeze(0)),
            y_c.to(device).unsqueeze(0),
        ).item()
    model.train()

    # Achieved εα for q=1 (closed form)
    eps_alpha_achieved = (steps_done * alpha) / (2.0 * (sigma ** 2))

    dp_sgd_inputs = dict(
        alpha=float(alpha),
        eps_alpha_target=float(eps_alpha),
        eps_alpha_achieved=float(eps_alpha_achieved),
        noise_multiplier=float(sigma),
        max_grad_norm=float(max_grad_norm),
        sample_rate=float(q),
        steps_planned=int(steps_planned),
        steps_done=int(steps_done),
        logical_batch_size=int(len(dataset)),  # THIS is the actual batch size used
        epochs=int(epochs),
        lr=float(lr),
        loss_reduction=loss_reduction,
    )

    return canary_loss, train_losses, dp_sgd_inputs


In [16]:
from pathlib import Path
from tqdm import trange
import numpy as np
import torch
import json
from torch.utils.data import TensorDataset

def run_50k_models_rdp(
    *,
    base_model,
    X_out, y_out,
    X_in, y_in,
    x_canary, y_canary,
    rdp_order: float,
    eps_alpha: float,
    max_grad_norm: float,
    epochs: int,
    lr: float,
    device: str,
    n_models_total: int = 50_000,
    save_dir: str = "exp_data_rdp_50k",
    save_every: int = 200,
    loss_reduction: str = "sum",
):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    R = n_models_total // 2

    dataset_out = TensorDataset(X_out.detach().cpu(), y_out.detach().cpu())
    dataset_in  = TensorDataset(X_in.detach().cpu(),  y_in.detach().cpu())
    canary_xy   = (x_canary.detach().cpu(), y_canary.detach().cpu())

    out_path = save_dir / "losses_out.npy"
    in_path  = save_dir / "losses_in.npy"

    losses_out = np.load(out_path).tolist() if out_path.exists() else []
    losses_in  = np.load(in_path).tolist()  if in_path.exists()  else []

    q = 1.0
    steps_planned = int(epochs)
    sigma = sigma_from_rdp_fullbatch(alpha=rdp_order, eps_alpha=eps_alpha, steps=steps_planned)

    dp_cfg = dict(
        rdp_order=float(rdp_order),
        eps_alpha_target=float(eps_alpha),
        noise_multiplier=float(sigma),
        max_grad_norm=float(max_grad_norm),
        sample_rate=float(q),
        steps_planned=int(steps_planned),
        epochs=int(epochs),
        lr=float(lr),
        loss_reduction=loss_reduction,
        logical_batch_size_out=int(len(dataset_out)),
        logical_batch_size_in=int(len(dataset_in)),
        n_out=int(len(dataset_out)),
        n_in=int(len(dataset_in)),
    )
    json.dump(dp_cfg, open(save_dir / "dp_sgd_inputs.json", "w"), indent=2)

    # OUT models
    for rep in trange(len(losses_out), R, desc="OUT models"):
        seed = 123456 + rep
        torch.manual_seed(seed)
        np.random.seed(seed)

        canary_loss, _, _ = dpsgd_rdp_run_fullbatch(
            base_model, dataset_out,
            canary_xy=canary_xy,
            alpha=rdp_order, eps_alpha=eps_alpha,
            max_grad_norm=max_grad_norm,
            epochs=epochs, lr=lr,
            device=device,
            noise_multiplier=sigma,
            loss_reduction=loss_reduction,
            return_train_losses=False,
        )
        losses_out.append(-float(canary_loss))

        if (rep + 1) % save_every == 0:
            np.save(out_path, np.asarray(losses_out, dtype=np.float32))

    # IN models
    for rep in trange(len(losses_in), R, desc="IN models"):
        seed = 789012 + rep
        torch.manual_seed(seed)
        np.random.seed(seed)

        canary_loss, _, _ = dpsgd_rdp_run_fullbatch(
            base_model, dataset_in,
            canary_xy=canary_xy,
            alpha=rdp_order, eps_alpha=eps_alpha,
            max_grad_norm=max_grad_norm,
            epochs=epochs, lr=lr,
            device=device,
            noise_multiplier=sigma,
            loss_reduction=loss_reduction,
            return_train_losses=False,
        )
        losses_in.append(-float(canary_loss))

        if (rep + 1) % save_every == 0:
            np.save(in_path, np.asarray(losses_in, dtype=np.float32))

    np.save(out_path, np.asarray(losses_out, dtype=np.float32))
    np.save(in_path,  np.asarray(losses_in,  dtype=np.float32))

    return np.asarray(losses_out, dtype=np.float32), np.asarray(losses_in, dtype=np.float32)


In [17]:
import torch
from torchvision import datasets, transforms

def load_data(dataset_name: str,
              data_dir: str | None = None,
              *,
              device: str | torch.device = "cpu",
              split: str = "train"):
    """
    Minimal loader that returns:
      X: torch.FloatTensor on `device` with shape (N, C, H, W)
      y: torch.LongTensor  on `device` with shape (N,)
      out_dim: int (number of classes)

    Supported dataset_name: "mnist", "fashionmnist", "cifar10"
    split: "train" or "test"
    """
    name = dataset_name.lower()
    root = data_dir if data_dir is not None else "./data"
    train = (split.lower() == "train")

    tfm = transforms.ToTensor()

    if name == "mnist":
        ds = datasets.MNIST(root=root, train=train, download=True, transform=tfm)
        out_dim = 10
    elif name in ("fashionmnist", "fashion-mnist", "fmnist"):
        ds = datasets.FashionMNIST(root=root, train=train, download=True, transform=tfm)
        out_dim = 10
    elif name == "cifar10":
        ds = datasets.CIFAR10(root=root, train=train, download=True, transform=tfm)
        out_dim = 10
    else:
        raise ValueError(f"Unsupported dataset_name={dataset_name!r}. Try mnist/fashionmnist/cifar10.")

    # Fast path: many torchvision datasets expose .data and .targets
    # MNIST/FashionMNIST: .data is uint8 (N,H,W), CIFAR10: .data is uint8 numpy (N,H,W,C)
    data = ds.data
    targets = ds.targets

    if isinstance(data, torch.Tensor):
        # MNIST-style: (N,H,W)
        if data.ndim == 3:
            X = data.unsqueeze(1)  # (N,1,H,W)
        else:
            X = data
        X = X.float() / 255.0
    else:
        # CIFAR10-style numpy: (N,H,W,C)
        X = torch.from_numpy(data).permute(0, 3, 1, 2).float() / 255.0  # (N,C,H,W)

    y = targets if isinstance(targets, torch.Tensor) else torch.tensor(targets)
    y = y.long()

    X = X.to(device)
    y = y.to(device)

    return X, y, out_dim

"""Auditing DP-SGD in black-box setting"""
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import os
import numpy as np
import argparse
from opacus.accountants.utils import get_noise_multiplier
import copy
from torch.utils.data import TensorDataset, DataLoader
import dill

from models import Models
from utils.data import load_data
from utils.dpsgd import clip_and_accum_grads
from utils.audit import compute_eps_lower_from_mia
from utils.clipbkd import craft_clipbkd, choose_worstcase_label

def xavier_init_model(model):
    """Initialize model using Xavier initialization"""
    def init_weights(m):
        if isinstance(m, nn.Linear) or isinstance(m, nn.Conv2d):
            torch.nn.init.xavier_normal_(m.weight)
            m.bias.data.fill_(0.01)

    model.apply(init_weights)


In [18]:
import torch

def make_neighbor_worlds_from_pool(
    X_pool: torch.Tensor,
    y_pool: torch.Tensor,
    *,
    n_in: int,
    x_canary: torch.Tensor,   # shape (1, C, H, W) (or (1, d))
    y_canary: torch.Tensor,   # shape (1,)
    seed: int = 0,
):
    """
    Build OUT world dataset D (size n_in-1) and IN world dataset D' = D ∪ {canary} (size n_in).

    This matches Algorithm 2 style: sample D of size (n-1), add target -> size n. :contentReference[oaicite:1]{index=1}
    """
    if n_in < 2:
        raise ValueError("n_in must be >= 2 (since OUT has n_in-1 points)")
    if (n_in - 1) > len(X_pool):
        raise ValueError(f"Need at least n_in-1={n_in-1} points in X_pool, but len(X_pool)={len(X_pool)}")

    # deterministic subset for reproducibility (fixed D across all trained models)
    g = torch.Generator(device="cpu")
    g.manual_seed(seed)
    idx = torch.randperm(len(X_pool), generator=g)[: (n_in - 1)]

    X_out = X_pool[idx]
    y_out = y_pool[idx]

    # IN world = OUT + canary
    X_in = torch.cat([X_out, x_canary], dim=0)
    y_in = torch.cat([y_out, y_canary], dim=0)

    return X_out, y_out, X_in, y_in


In [19]:
import torch
import numpy as np
from torch.utils.data import TensorDataset
from utils.clipbkd import craft_clipbkd, choose_worstcase_label
from models import Models

device = "cuda:0" if torch.cuda.is_available() else "cpu"

dataset = "mnist"
model_type = "cnn"
canary = "blank"

X_all, y_all, out_dim = load_data(dataset, None, device=device, split="train")


# Choose the pool of data you want to sample from
# If you want to mimic the paper's "half for DP-SGD, half for pretraining" idea,
# set X_pool = first half and keep the second half for your non-private pretraining.
X_pool, y_pool = X_all, y_all   # simplest: use all MNIST training points

# Canary: Blank
target_X = torch.zeros_like(X_all[[0]])                 # (1, C, H, W)
target_y = torch.tensor([9], device=device).long()      # (1,)


# ---- HERE is the dataset size knob like Fig 3 ----
n_in = 100   # try 100, 1000, or len(X_pool) (or 30000 if you use a half-split)
seed_dataset = 0

X_out, y_out, X_in, y_in = make_neighbor_worlds_from_pool(
    X_pool, y_pool,
    n_in=n_in,
    x_canary=target_X,
    y_canary=target_y,
    seed=seed_dataset,

)

# Define base init model (must be plain nn.Module)
init_model = Models[model_type](X_out.shape, out_dim=out_dim).to(device)
xavier_init_model(init_model)  # or init_model.load_state_dict(torch.load(...))

#comment out for blank canary keep for clipbkd

#init_model.eval()  # safe default; ClipBKD code may not care

# Craft ClipBKD canary using OUT-world data + fixed init θ0
if canary == "clipbkd":
    target_X, target_y = craft_clipbkd(X_out, init_model, device)

    # Some implementations return target_y already; if not, uncomment:
    # target_y = choose_worstcase_label(init_model, target_X)

    # ---- shape hygiene (important!) ----
    # Make sure canary is (1, ...) and label is (1,)
    if target_X.ndim == X_out.ndim - 1:
        target_X = target_X.unsqueeze(0)
    if target_y.ndim == 0:
        target_y = target_y.unsqueeze(0)

    target_X = target_X.to(device)
    target_y = target_y.to(device).long()

    # IN world dataset = OUT + canary
    X_in = torch.cat([X_out, target_X], dim=0)
    y_in = torch.cat([y_out, target_y], dim=0)


In [20]:
rdp_order = 2.0
eps_alpha = 5.0
n_models_total = 100

max_grad_norm = 1.0
epochs = 50
lr = 0.01 #should be custom
out_dir = "RDP_exp_data"

losses_out, losses_in = run_50k_models_rdp(
    base_model=init_model,
    X_out=X_out, y_out=y_out,
    X_in=X_in,   y_in=y_in,
    x_canary=target_X.squeeze(0),
    y_canary=target_y.squeeze(0),
    rdp_order=rdp_order,
    eps_alpha=eps_alpha,
    max_grad_norm=max_grad_norm,
    epochs=epochs,
    lr=lr,
    device=device,
    n_models_total=n_models_total,
    save_dir=f"{out_dir}/{dataset}_{model_type}_eps_{eps_alpha}_a_{rdp_order}_dataset_{n_in}_canary_{canary}",
    save_every=200,
    loss_reduction="sum",
)

print(losses_out[:1000], losses_in[:1000])
print("std out/in:", losses_out.std(), losses_in.std())

OUT models: 0it [00:00, ?it/s]
IN models: 0it [00:00, ?it/s]

[-2.8264503  -3.638711   -5.2369     -0.6237148  -6.564086   -5.0276866
 -3.8050656  -3.5985692  -2.4021025  -2.0987597  -2.7548301  -4.1343174
 -2.8667598  -3.7045808  -0.18216701 -1.3592594  -4.078909   -3.2046905
 -4.0533466  -3.8294294  -4.4895473  -2.7215285  -5.694233   -4.118522
 -2.0251102  -4.3753157  -4.570836   -3.4076219  -3.5472867  -3.2123048
 -2.1943352  -5.0521307  -2.5123205  -3.9953492  -4.151149   -5.810842
 -3.9793901  -2.9487648  -3.0785387  -5.506897   -1.9616134  -6.3298492
 -3.0361714  -1.7899646  -2.37423    -2.1147323  -4.2921815  -4.9791403
 -5.162686   -5.108525  ] [-1.5146704  -1.2441272  -0.99046314 -1.6264455  -1.4976661  -0.6029748
 -1.198845   -0.2224605  -2.361195   -2.7623038  -0.90369457 -1.5887644
 -1.983366   -3.7268057  -2.4373639  -2.4398386  -0.45549357 -1.2560707
 -1.6372474  -1.0524276  -0.755348   -1.9497356  -1.6812092  -1.788778
 -0.89652675 -1.1528578  -1.8310587  -0.7476112  -1.1981509  -1.1084163
 -0.5974555  -0.8467909  -0.89159524 -0.8

In [21]:
print(losses_out[:1000])

[-2.8264503  -3.638711   -5.2369     -0.6237148  -6.564086   -5.0276866
 -3.8050656  -3.5985692  -2.4021025  -2.0987597  -2.7548301  -4.1343174
 -2.8667598  -3.7045808  -0.18216701 -1.3592594  -4.078909   -3.2046905
 -4.0533466  -3.8294294  -4.4895473  -2.7215285  -5.694233   -4.118522
 -2.0251102  -4.3753157  -4.570836   -3.4076219  -3.5472867  -3.2123048
 -2.1943352  -5.0521307  -2.5123205  -3.9953492  -4.151149   -5.810842
 -3.9793901  -2.9487648  -3.0785387  -5.506897   -1.9616134  -6.3298492
 -3.0361714  -1.7899646  -2.37423    -2.1147323  -4.2921815  -4.9791403
 -5.162686   -5.108525  ]


In [32]:
from pathlib import Path
import torch

from models import Models
from utils.data import load_data
from utils.clipbkd import craft_clipbkd, choose_worstcase_label

device = "cuda:0" if torch.cuda.is_available() else "cpu"

def maybe_skip_run(save_dir: str) -> bool:
    p = Path(save_dir)
    return (p / "losses_out.npy").exists() and (p / "losses_in.npy").exists()

def build_canary(canary_type: str, X_all: torch.Tensor, init_model, X_out: torch.Tensor):
    if canary_type == "blank":
        target_X = torch.zeros_like(X_all[[0]])                 # (1, ...)
        target_y = torch.tensor([9], device=device).long()      # (1,)
        return target_X.to(device), target_y.to(device)

    if canary_type == "clipbkd":
        #init_model.eval()
        target_X, target_y = craft_clipbkd(X_out, init_model, device)

        # If needed:
        # target_y = choose_worstcase_label(init_model, target_X)

        # Shape hygiene
        if target_X.ndim == X_out.ndim - 1:
            target_X = target_X.unsqueeze(0)
        if target_y.ndim == 0:
            target_y = target_y.unsqueeze(0)

        return target_X.to(device), target_y.to(device).long()

    raise ValueError(f"Unknown canary_type: {canary_type}")

def run_sweep(
    out_dir="data_1_12",
    datasets=("mnist", "cifar10"),
    model_types=("cnn", "lr"),
    eps_list=(1.0, 4.0, 8.0),
    rdp_orders=(2.0, 5.0),
    dataset_sizes=("100", "1000", "10000", "full"),
    n_models_total=20000,
    max_grad_norm=1.0,
    seed_base=0,
    loss_reduction="sum",
    save_every=200,
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # ✅ canary policy per model
    canary_plan = {
        "cnn": ("blank", "clipbkd"),
        "lr":  ("clipbkd",),
    }

    for n_in_tag in dataset_sizes:  # OUTERMOST LOOP: dataset size
        for dataset in datasets:
            X_all, y_all, out_dim = load_data(dataset, None, device=device, split="train")
            X_pool, y_pool = X_all, y_all

            if n_in_tag == "full":
                n_in = len(X_pool)
            else:
                n_in = int(n_in_tag)

            for model_type in model_types:
                if model_type == "cnn":
                    epochs = 5
                    lr = 1/n_in
                elif model_type == "lr":
                    epochs = 5
                    lr = 1/n_in
                else:
                    raise ValueError(f"Unknown model_type: {model_type}")

                for eps_alpha in eps_list:
                    for rdp_order in rdp_orders:
                        # per-config seed (stable-ish)
                        seed_dataset = seed_base
                        seed_dataset += (hash(dataset) % 10_000) * 17
                        seed_dataset += (hash(model_type) % 10_000) * 19
                        seed_dataset += int(float(eps_alpha) * 10) * 23
                        seed_dataset += int(float(rdp_order) * 10) * 29
                        seed_dataset += (0 if n_in_tag == "full" else n_in) * 31
                        seed_dataset = seed_dataset % (2**31 - 1)

                        # Build OUT world first (using a placeholder canary if your helper needs it)
                        placeholder_X = torch.zeros_like(X_all[[0]]).to(device)
                        placeholder_y = torch.tensor([9], device=device).long()

                        X_out, y_out, _, _ = make_neighbor_worlds_from_pool(
                            X_pool, y_pool,
                            n_in=n_in,
                            x_canary=placeholder_X,
                            y_canary=placeholder_y,
                            seed=seed_dataset,
                        )

                        # Init model on OUT-world shape
                        init_model = Models[model_type](X_out.shape, out_dim=out_dim).to(device)
                        xavier_init_model(init_model)

                        # ✅ canary loop depends on model type
                        for canary_type in canary_plan[model_type]:
                            target_X, target_y = build_canary(
                                canary_type=canary_type,
                                X_all=X_all,
                                init_model=init_model,
                                X_out=X_out,
                            )

                            # IN world = OUT + canary (explicit, always)
                            X_in = torch.cat([X_out, target_X], dim=0)
                            y_in = torch.cat([y_out, target_y], dim=0)

                            save_dir = out_dir / (
                                f"{dataset}_{model_type}"
                                f"_eps_{eps_alpha}_a_{rdp_order}"
                                f"_dataset_{n_in_tag}"
                                f"_canary_{canary_type}"
                            )
                            save_dir_str = str(save_dir)

                            if maybe_skip_run(save_dir_str):
                                print(f"[skip] {save_dir_str}")
                                continue

                            print(
                                f"[run] size={n_in_tag} dataset={dataset} model={model_type} "
                                f"canary={canary_type} eps={eps_alpha} a={rdp_order} "
                                f"epochs={epochs} lr={lr} seed={seed_dataset}"
                            )

                            try:
                                losses_out, losses_in = run_50k_models_rdp(
                                    base_model=init_model,
                                    X_out=X_out, y_out=y_out,
                                    X_in=X_in,   y_in=y_in,
                                    x_canary=target_X.squeeze(0),
                                    y_canary=target_y.squeeze(0),
                                    rdp_order=rdp_order,
                                    eps_alpha=eps_alpha,
                                    max_grad_norm=max_grad_norm,
                                    epochs=epochs,
                                    lr=lr,
                                    device=device,
                                    n_models_total=n_models_total,
                                    save_dir=save_dir_str,
                                    save_every=save_every,
                                    loss_reduction=loss_reduction,
                                )
                                print(
                                    f"  done: std(out)={losses_out.std():.4g} "
                                    f"std(in)={losses_in.std():.4g}"
                                )
                            except Exception as e:
                                print(f"[ERROR] {save_dir_str}\n  {repr(e)}")


# run it
run_sweep(
    out_dir="data_1_12",
    datasets=("mnist", "cifar10"),
    model_types=("cnn", "lr"),
    eps_list=(1.0, 4.0, 10.0, 2.0),
    rdp_orders=(2.0, 5.0),
    dataset_sizes=("full", "1000", "100"),
)


[run] size=full dataset=mnist model=cnn canary=blank eps=1.0 a=2.0 epochs=5 lr=1.6666666666666667e-05 seed=164653


OUT models:   0%|          | 0/10000 [00:00<?, ?it/s]/raid/bdkim4/conda/envs/bb_audit_dpsgd/lib/python3.10/site-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
IN models:  61%|██████    | 6122/10000 [8:03:23<4:57:19,  4.60s/it] 